# Assignment 1B: domain LLM adaptation and production optimization

**Domain data → QLoRA fine-tuning → decoding, speculative decoding, 4-bit → cost per 1M tokens · 15 marks**

| | |
|---|---|
| **Domain** | Variant 5: IT Helpdesk & Incident Resolution (RHEL 9 + Ubuntu Server guides), the same corpus as 1A |
| **Target model** | `Qwen/Qwen2.5-1.5B` (base, 1.54B parameters) |
| **Draft model** (speculative decoding) | `Qwen/Qwen2.5-0.5B`: same tokenizer, 3× smaller |
| **Run on** | Any NVIDIA GPU: Colab T4, the institute GPU server, or A100/L40S. The setup cell detects where it is running. |

Open **`explainer.html`** first. It explains every idea here and **how this notebook is organised like a production
experiment** (fixed prompts, one config, a baseline first, one change at a time, honest benchmarking).

This notebook is complete and runnable top to bottom. Your work is to **run it, read the outputs, and write the ✍️
inference cells**. Marks depend on those, and each one has a hidden "what to look for" hint.

| Part | Step | Marks |
|---|---|---|
| A | 1 Corpus collection and cleaning | 1 |
| A | 2 Baseline outputs + architecture | 1 |
| B | B1 Instruction dataset | 2 |
| B | B2 QLoRA (Adapter B) | 2 |
| B | B3 Baseline vs adapter | 1 |
| C | 1 Five decoding strategies | 3 |
| C | 2 Speculative decoding | 2 |
| C | 3 4-bit quantization + cost per 1M tokens | 3 |

**Run order and time** (T4): Part A ≈ 10 min · Part B ≈ 30 min · Part C ≈ 25 min. A100: about a third of that.

---
## Cell 0: Setup and the experiment contract

In production, everything that could change a result is fixed in **one place** before any experiment runs:
model IDs, prompts, seeds, generation length, prices. Every number later in this notebook can be traced back to this cell.

In [ ]:
%pip install -q -U transformers peft trl bitsandbytes accelerate datasets pymupdf langdetect

In [ ]:
import os, re, gc, sys, json, math, time, random, hashlib, collections, statistics, shutil
from pathlib import Path
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
pd.set_option('display.max_colwidth', None)

# ---- where are we running? -----------------------------------------------
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive')     # survives Colab disconnects
else:
    ROOT = Path.home()                        # institute GPU server / rented machine
WORKDIR = ROOT / 'llm_assignment_1b'

# reuse the PDFs downloaded in Assignment 1A if they exist (no point downloading 25 PDFs twice)
PDF_DIR_1A = ROOT / 'llm_assignment_1a' / 'domain_pdfs'
PDF_DIR    = PDF_DIR_1A if PDF_DIR_1A.exists() and any(PDF_DIR_1A.glob('*.pdf')) else WORKDIR / 'domain_pdfs'
CORPUS_DIR = WORKDIR / 'domain_corpus'          # deliverable 4
DATA_DIR   = WORKDIR / 'data'                   # jsonl, csv, json results
ADAPTER_DIR = WORKDIR / 'qlora_adapter_B'
for d in [PDF_DIR, CORPUS_DIR, DATA_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ---- hardware -------------------------------------------------------------
assert torch.cuda.is_available(), 'No GPU. Colab: Runtime > Change runtime type > GPU.'
GPU_NAME = torch.cuda.get_device_name(0)
BF16_OK  = torch.cuda.get_device_capability()[0] >= 8      # Ampere+ (A100, L4, L40S)
DTYPE16  = torch.bfloat16 if BF16_OK else torch.float16   # the "16-bit baseline" dtype
HOURLY_RATE = 12.00 if 'A100' in GPU_NAME else 3.50       # from the assignment; see Part C3

# ---- the experiment contract ---------------------------------------------
CONFIG = {
    'target_model': 'Qwen/Qwen2.5-1.5B',
    'draft_model':  'Qwen/Qwen2.5-0.5B',
    'seed': 42,
    'max_new_tokens': 150,
    'prompt_template': 'Question: {q}\nAnswer:',      # base models: no system prompt, plain Q/A framing
    'gpu': GPU_NAME,
    'dtype_16bit': str(DTYPE16),
    'hourly_rate': HOURLY_RATE,
}
SEED = CONFIG['seed']
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DOMAIN_PROMPTS = [                                   # used in A2, B3, C1, C2
    'How do I permanently open TCP port 8080 in firewalld on RHEL 9?',
    'How do I configure a static IPv4 address on a network interface using nmcli?',
    'How can I find out which package provides a specific file using dnf?',
]
BENCH_PROMPTS = DOMAIN_PROMPTS + [                   # 10 prompts for C3
    'How do I check the current SELinux mode and switch it to permissive temporarily?',
    'How do I create a logical volume with LVM and extend it later?',
    'How do I enable a systemd service so that it starts at boot?',
    'How do I list the ports that are open in the public zone of firewalld?',
    'How do I mount an NFS share automatically at boot?',
    'How do I see which kernel version is running and which kernels are installed?',
    'How do I give a user sudo rights on RHEL by adding them to the wheel group?',
]
CONFIG['domain_prompts'] = DOMAIN_PROMPTS
CONFIG['bench_prompts'] = BENCH_PROMPTS

import transformers, peft, trl, bitsandbytes
CONFIG['versions'] = {m.__name__: m.__version__ for m in [torch, transformers, peft, trl, bitsandbytes]}
print(json.dumps({k: v for k, v in CONFIG.items() if 'prompts' not in k}, indent=1))
print('PDFs from:', PDF_DIR)

---
# PART A: Domain data collection and baseline

## Step 1: Corpus collection and cleaning [1 mark] · *explainer §3*

The same pipeline as 1A, written out in full: **extract page by page → normalise → length filter → boilerplate/TOC filter
→ exact dedup → language filter**. The assignment asks for at least 5 PDFs; we use 25, because a bigger corpus means more
instruction pairs in B1.

Design choices you will justify in the inference:
- **Unit = page.** A whole manual is too coarse (one bad page would kill 100 good ones); a line is too fine.
- **Length threshold = 50 words.** About 3–4 sentences. It removes title, blank and divider pages, and keeps short real sections.
- **Dedup = md5 hash** of lower-cased, punctuation-stripped text: an exact-duplicate detector.
- **Language = `langdetect`** (a port of Google's language-detection library) on the first 2,000 characters, with a fixed seed.

In [ ]:
import urllib.request
RHEL = 'https://docs.redhat.com/en/documentation/red_hat_enterprise_linux/9/pdf/{s}/Red_Hat_Enterprise_Linux-9-{S}-en-US.pdf'
RHEL_GUIDES = [
    'configuring_basic_system_settings', 'configuring_firewalls_and_packet_filters',
    'configuring_and_managing_networking', 'managing_storage_devices', 'managing_file_systems',
    'configuring_and_managing_logical_volumes', 'managing_software_with_the_dnf_tool',
    'securing_networks', 'security_hardening', 'using_selinux',
    'configuring_authentication_and_authorization_in_rhel', 'managing_monitoring_and_updating_the_kernel',
    'monitoring_and_managing_system_status_and_performance', 'deploying_web_servers_and_reverse_proxies',
    'managing_networking_infrastructure_services', 'configuring_and_using_network_file_services',
    'automating_system_administration_by_using_rhel_system_roles', 'managing_systems_using_the_rhel_9_web_console',
    'building_running_and_managing_containers', 'configuring_and_managing_virtualization',
    'managing_and_monitoring_security_updates', 'upgrading_from_rhel_8_to_rhel_9',
    'deploying_mail_servers', 'configuring_and_using_database_servers',
]
SOURCES = {f'rhel9_{s}': RHEL.format(s=s, S=s[0].upper() + s[1:]) for s in RHEL_GUIDES}
SOURCES['ubuntu_server_guide'] = 'https://documentation.ubuntu.com/server/_/downloads/en/latest/pdf/'

for name, url in SOURCES.items():                      # idempotent: skips files that already exist
    out = PDF_DIR / f'{name}.pdf'
    if out.exists() and out.stat().st_size > 10_000:
        continue
    try:
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        out.write_bytes(urllib.request.urlopen(req, timeout=120).read())
        print('downloaded', name)
    except Exception as e:
        print('FAILED', name, e)
pdfs = sorted(PDF_DIR.glob('*.pdf'))
print(f'{len(pdfs)} PDFs, {sum(p.stat().st_size for p in pdfs) / 1e6:.1f} MB')

In [ ]:
import fitz  # PyMuPDF
from langdetect import detect, DetectorFactory
DetectorFactory.seed = 0

def extract_pages(pdf_path):
    with fitz.open(pdf_path) as doc:
        return [page.get_text() for page in doc]

BOILER_LINE = [re.compile(p) for p in [r'^\d{1,4}$', r'^CHAPTER \d+\. ', r'^APPENDIX [A-Z]\. ']]
def normalise_doc(pages):
    # remove lines repeated on >=30% of pages (running headers/footers), re-join wrapped lines
    lines_per_page = [[l.strip() for l in p.splitlines() if l.strip()] for p in pages]
    counts = collections.Counter(l for ls in lines_per_page for l in set(ls))
    threshold = max(3, 0.3 * len(pages))
    cleaned = []
    for ls in lines_per_page:
        keep = [l for l in ls if counts[l] < threshold and not any(b.match(l) for b in BOILER_LINE)]
        merged = []
        for l in keep:
            if merged and not re.search(r'[.:;!?]$', merged[-1]) and re.match(r'[a-z(]', l):
                merged[-1] += ' ' + l
            else:
                merged.append(l)
        cleaned.append(re.sub(r'[ \t]+', ' ', '\n'.join(merged)))
    return cleaned

MIN_WORDS = 50
HEADING = re.compile(r'^\d+(?:\.\d+)+\.?\s+[A-Za-z]')
BOILER_PHRASES = ('PROVIDING FEEDBACK ON', 'Legal Notice', 'MAKING OPEN SOURCE MORE INCLUSIVE', 'Table of Contents')
def is_boilerplate(t):
    ls = t.splitlines()
    return sum(bool(HEADING.match(l)) for l in ls) / max(1, len(ls)) > 0.3 or any(b in t for b in BOILER_PHRASES)
def fingerprint(t):
    return hashlib.md5(re.sub(r'\W+', ' ', t.lower()).strip().encode()).hexdigest()
def is_english(t):
    try:
        return detect(t[:2000]) == 'en'
    except Exception:
        return False

raw_pages = {p.stem: extract_pages(p) for p in pdfs}
pages = [(d, i, t) for d, ps in raw_pages.items() for i, t in enumerate(normalise_doc(ps))]
stats = []
def record(step, rows):
    stats.append({'step': step, 'PDFs / documents': len({r[0] for r in rows}), 'pages': len(rows),
                  'words': sum(len(r[2].split()) for r in rows)})

record('0 raw extracted', pages)
pages = [r for r in pages if len(r[2].split()) >= MIN_WORDS];  record('1 length >= 50 words', pages)
pages = [r for r in pages if not is_boilerplate(r[2])];        record('2 boilerplate / TOC', pages)
seen, uniq = set(), []
for r in pages:
    h = fingerprint(r[2])
    if h not in seen:
        seen.add(h); uniq.append(r)
pages = uniq;                                                  record('3 exact dedup (md5)', pages)
pages = [r for r in pages if is_english(r[2])];                record('4 language = en', pages)

corpus = collections.defaultdict(list)
for d, i, t in pages:
    corpus[d].append(t)
for f in CORPUS_DIR.glob('*.txt'):
    f.unlink()
for d, ps in corpus.items():
    (CORPUS_DIR / f'{d}.txt').write_text('\n\n'.join(ps), encoding='utf-8')

stats_df = pd.DataFrame(stats)
stats_df['pages removed'] = (-stats_df['pages'].diff()).fillna(0).astype(int)
stats_df['% of raw removed'] = (100 * stats_df['pages removed'] / stats_df['pages'].iloc[0]).round(1)
stats_df.to_csv(DATA_DIR / 'cleaning_stats.csv', index=False)
print(f'{len(corpus)} cleaned .txt files in {CORPUS_DIR}')
stats_df

#### ✍️ Your inference (mandatory, write it in your own words)

- Corpus statistics before/after each step (#PDFs, #.txt files, pages): refer to the table.
- Which step reduced the corpus the most, and why does a set of product manuals have that problem?
- Justify the 50-word threshold, the dedup method (md5 exact hash) and the language tool (langdetect).

<details><summary>What to look for in your outputs</summary>

- Look at the 'pages removed' column. Manuals have many short divider/title pages, and a TOC + legal + feedback block in every PDF.
- Exact hashing misses near-duplicates (the same legal notice with a different product name). MinHash / Jaccard would catch those.
- The language filter removes almost nothing, because official English docs are already English. Saying so is a valid inference.
</details>

_Write your answer here._

## Step 2: Baseline model output + architecture [1 mark] · *explainer §4*

We load the **target model in 16-bit** (bf16 where the GPU supports it, fp16 on a T4, which has no bf16; this is inference
only, so fp16 is safe here). There is no system prompt, and every prompt goes through `CONFIG['prompt_template']`.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

tok = AutoTokenizer.from_pretrained(CONFIG['target_model'])
tok.padding_side = 'left'
IM_END = tok.convert_tokens_to_ids('<|im_end|>')
STOP_IDS = [tok.eos_token_id, IM_END]

def free(*names):
    # delete global models by name and release GPU memory: needed between benchmarks
    for n in names:
        globals().pop(n, None)
    gc.collect(); torch.cuda.empty_cache()

def fmt(q, chat=False):
    if chat:
        return tok.apply_chat_template([{'role': 'user', 'content': q}], tokenize=False, add_generation_prompt=True)
    return CONFIG['prompt_template'].format(q=q)

@torch.no_grad()
def run(model, prompt, gen=None, chat=False, assistant=None, seed=SEED):
    # one timed generation -> (text, new_tokens, seconds). Timing is done correctly: see explainer §8.
    gen = dict(do_sample=False) if gen is None else gen
    enc = tok(fmt(prompt, chat), return_tensors='pt').to(model.device)
    torch.manual_seed(seed)
    torch.cuda.synchronize(); t0 = time.perf_counter()
    out = model.generate(**enc, max_new_tokens=CONFIG['max_new_tokens'], eos_token_id=STOP_IDS,
                         pad_token_id=tok.eos_token_id, assistant_model=assistant, **gen)
    torch.cuda.synchronize(); dt = time.perf_counter() - t0
    new = out[0, enc['input_ids'].shape[1]:]
    return tok.decode(new, skip_special_tokens=True).strip(), int(new.numel()), dt

model = AutoModelForCausalLM.from_pretrained(CONFIG['target_model'], dtype=DTYPE16, device_map={'': 0})
c = model.config
arch = pd.DataFrame([
    ['total parameters', f'{sum(p.numel() for p in model.parameters()):,}'],
    ['decoder layers', c.num_hidden_layers],
    ['hidden size', c.hidden_size],
    ['attention heads / KV heads', f'{c.num_attention_heads} / {c.num_key_value_heads}'],
    ['vocabulary size (config / tokenizer)', f'{c.vocab_size:,} / {len(tok):,}'],
    ['max context', f'{c.max_position_embeddings:,}'],
    ['dtype / memory', f'{model.dtype} / {model.get_memory_footprint() / 1e9:.2f} GB'],
], columns=['property', 'value'])
display(arch)

run(model, 'warm-up')                       # the first CUDA call is always slow: never time it
baseline = pd.DataFrame([{'prompt': p, 'baseline_output': run(model, p)[0]} for p in DOMAIN_PROMPTS])
baseline.to_csv(DATA_DIR / 'baseline_outputs.csv', index=False)
baseline

#### ✍️ Your inference (mandatory, write it in your own words)

- Report parameters, layers, hidden size and vocabulary size.
- Judge the 3 baseline answers: correct commands? complete? any invented flags (hallucinations)? Does it stop, or ramble into new questions?

<details><summary>What to look for in your outputs</summary>

- A base model often answers the first line well, then writes another 'Question:' and keeps going. That is the 'continues text' behaviour from CS-2 slide 27.
- Check specific flags: firewalld needs --permanent and --reload; nmcli needs ipv4.method manual; dnf provides needs a path or a glob.
</details>

_Write your answer here._

---
# PART B: Instruction fine-tuning with QLoRA

## B1: Instruction dataset [2 marks] · *explainer §5*

**Method: heuristic template conversion** (CS-3 slide 8). Every numbered manual section (`1.7.3. Assigning a network
interface to a zone`) becomes one pair: the title becomes a question, and the first ~160 words of the section become the
answer. That keeps every answer **grounded** in the corpus. It is free, deterministic and needs no API key.

**Size justification:** for a 1.5B model on a T4, about 2,000 pairs × ~250 tokens is ~0.5M tokens per epoch. Two epochs take about
20–25 min. More pairs give diminishing returns for a small adapter, and fewer pairs underfit.

In [ ]:
SECTION = re.compile(r'^(\d+(?:\.\d+)+)\.?\s+([A-Z][^\n]{3,100})$')
SKIP_TITLES = ('additional resources', 'prerequisites', 'verification', 'next steps', 'procedure')

def sections(text):
    secs, cur = [], None
    for line in text.splitlines():
        line = line.strip()
        m = SECTION.match(line)
        if m:
            cur = [m.group(2), []]; secs.append(cur)
        elif cur and not cur[1] and cur[0].isupper() and line.isupper():
            cur[0] += ' ' + line                           # ALL-CAPS title wrapped onto a second line
        elif cur:
            cur[1].append(line)
    return secs

def make_pair(title, words, product):
    title = title.capitalize() if title.isupper() else title
    t = title[0].lower() + title[1:]
    rng = random.Random(title)                             # same title -> same template: reproducible
    if title.split()[0].lower().endswith('ing'):
        q = rng.choice([f'Walk me through {t} on {product}.', f'What are the steps for {t} on {product}?',
                        f'How do I go about {t} on {product}?'])
    else:
        q = rng.choice([f'Explain {t} in {product}.', f'What should an admin know about {t} on {product}?'])
    a = ' '.join(words[:160])
    cut = a.rfind('. ')
    return q, (a[:cut + 1] if cut > 200 else a)

pairs = []
for f in sorted(CORPUS_DIR.glob('*.txt')):
    product = 'Ubuntu Server' if 'ubuntu' in f.stem else 'RHEL 9'
    for title, body in sections(f.read_text(encoding='utf-8')):
        words = ' '.join(body).split()
        if len(words) >= 40 and not title.lower().startswith(SKIP_TITLES):
            q, a = make_pair(title, words, product)
            pairs.append({'instruction': q, 'response': a})
seen_q = set()
pairs = [p for p in pairs if not (p['instruction'] in seen_q or seen_q.add(p['instruction']))]

MAX_PAIRS = 2000
random.Random(SEED).shuffle(pairs)                         # fixed seed -> the same split every run
pairs = pairs[:MAX_PAIRS]
with open(DATA_DIR / 'instruction_dataset.jsonl', 'w', encoding='utf-8') as f:
    for p in pairs:
        f.write(json.dumps(p, ensure_ascii=False) + '\n')
n_train = int(0.8 * len(pairs))
train_pairs, eval_pairs = pairs[:n_train], pairs[n_train:]

tok_lens = [len(tok(p['instruction'] + ' ' + p['response'])['input_ids']) for p in pairs]
print(f'pairs: {len(pairs)}  train: {len(train_pairs)}  eval: {len(eval_pairs)}  (seed {SEED})')
print(f'tokens per pair: mean {np.mean(tok_lens):.0f}, max {max(tok_lens)}  -> max_length 512 is enough')
CONFIG['dataset'] = {'pairs': len(pairs), 'train': len(train_pairs), 'eval': len(eval_pairs),
                     'mean_tokens': round(float(np.mean(tok_lens)))}
pd.DataFrame(pairs[:5])

<details><summary><b>Alternative: synthetic generation with an external LLM (exact prompt template)</b></summary>

Not used here, but this is the template you would use, one call per ~800-word chunk:
```
Read the text below and generate 10 instruction-response pairs in JSON format based ONLY on this text.
Each entry must have "instruction" and "response" keys. Instructions should read like real helpdesk
questions from a Linux administrator. Responses must use only facts, commands and file paths that
appear in the text. Return a JSON list and nothing else.

TEXT:
<chunk>
```
</details>

#### ✍️ Your inference (mandatory, write it in your own words)

- Method used and why; its weaknesses (look at the 5 samples above: wording, cut-off answers).
- Dataset size and average tokens per example, and why that size fits this model and GPU.
- Train/eval counts and the seed. How do you know no fact outside the corpus got in?

_Write your answer here._

## B2: QLoRA fine-tuning with Adapter B [2 marks] · *explainer §5*

The base model is frozen in **4-bit NF4**. **Adapter B** (`r=16, α=32, q_proj + v_proj`) is trained in 16-bit. Examples are
formatted with the model's **chat template** by `SFTTrainer`.

| Hyperparameter | Value | Why |
|---|---|---|
| batch × grad-accum | 8 × 2 = 16 | a stable gradient that fits in T4 memory |
| learning rate | 2e-4, cosine, 10 warmup steps | the standard LoRA LR: B starts at zero, so it needs bigger steps than full fine-tuning |
| epochs | 2 | enough to learn the format; the eval loss is checked for overfitting |
| max length | 512 tokens | longer than the longest pair (see B1) |
| LoRA dropout | 0.05 | light regularisation |

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from trl import SFTConfig, SFTTrainer
from datasets import Dataset

free('model')
bnb_cfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                             bnb_4bit_compute_dtype=DTYPE16, bnb_4bit_use_double_quant=True)
qmodel = AutoModelForCausalLM.from_pretrained(CONFIG['target_model'], quantization_config=bnb_cfg, device_map={'': 0})
qmodel = prepare_model_for_kbit_training(qmodel, use_gradient_checkpointing=True)

LORA = dict(r=16, lora_alpha=32, target_modules=['q_proj', 'v_proj'], lora_dropout=0.05)
qmodel = get_peft_model(qmodel, LoraConfig(**LORA, bias='none', task_type='CAUSAL_LM'))
qmodel.print_trainable_parameters()

to_chat = lambda p: {'messages': [{'role': 'user', 'content': p['instruction']},
                                  {'role': 'assistant', 'content': p['response']}]}
train_ds = Dataset.from_list([to_chat(p) for p in train_pairs])
eval_ds  = Dataset.from_list([to_chat(p) for p in eval_pairs])

SFT = dict(per_device_train_batch_size=8, gradient_accumulation_steps=2, num_train_epochs=2,
           learning_rate=2e-4, lr_scheduler_type='cosine', warmup_steps=10, max_length=512)
CONFIG['qlora'] = {**LORA, **SFT, 'quant': 'nf4 + double quant', 'compute_dtype': str(DTYPE16)}
args = SFTConfig(output_dir=str(WORKDIR / 'sft_runs'), **SFT, logging_steps=10, eval_strategy='epoch',
                 save_strategy='no', bf16=BF16_OK, fp16=not BF16_OK, gradient_checkpointing=True,
                 report_to='none', seed=SEED)
trainer = SFTTrainer(model=qmodel, args=args, train_dataset=train_ds, eval_dataset=eval_ds, processing_class=tok)
torch.cuda.reset_peak_memory_stats()
t0 = time.time(); trainer.train()
print(f'training: {(time.time() - t0) / 60:.1f} min, peak VRAM {torch.cuda.max_memory_allocated() / 1e9:.2f} GB')
qmodel.save_pretrained(ADAPTER_DIR)

hist = pd.DataFrame(trainer.state.log_history)
fig, ax = plt.subplots(figsize=(8, 3.5))
h = hist.dropna(subset=['loss']); ax.plot(h['step'], h['loss'], label='train loss')
e = hist.dropna(subset=['eval_loss']) if 'eval_loss' in hist else pd.DataFrame()
if len(e): ax.plot(e['step'], e['eval_loss'], 'o--', label='eval loss')
ax.set_xlabel('step'); ax.set_ylabel('loss'); ax.legend(); ax.grid(alpha=.3); plt.show()

#### ✍️ Your inference (mandatory, write it in your own words)

- Why Adapter B? Report trainable parameters and % (from print_trainable_parameters).
- Report the hyperparameters, training time and peak VRAM.
- Train vs eval loss: is it learning, underfitting or overfitting?

<details><summary>What to look for in your outputs</summary>

- Adapter B trains ~2.2M of 1.54B parameters (~0.14%).
- If the eval loss rises in epoch 2 while the train loss keeps falling, that's overfitting: use 1 epoch, or more data.
</details>

_Write your answer here._

## B3: Evaluation: baseline vs adapter [1 mark]
The same 3 prompts. The adapter uses the chat template it was trained with; the baseline used the plain Q/A template.

In [ ]:
qmodel.eval()                                  # turn off LoRA dropout for evaluation
qmodel.config.use_cache = True
run(qmodel, 'warm-up', chat=True)
baseline['adapter_B_output'] = [run(qmodel, p, chat=True)[0] for p in DOMAIN_PROMPTS]
baseline.to_csv(DATA_DIR / 'baseline_vs_adapter.csv', index=False)
free('qmodel', 'trainer')
baseline

#### ✍️ Your inference (mandatory, write it in your own words)

- Side by side, per prompt: correctness, domain terminology, completeness, hallucinations. What improved, and what got worse?

<details><summary>What to look for in your outputs</summary>

- Typical: the adapter answers in the manual's voice and stops cleanly; the baseline rambles. But the heuristic answers are cut at 160 words, so the adapter may copy that truncated, section-intro style rather than giving exact commands.
- A hallucinated flag or package name counts against correctness even if it sounds confident.
</details>

_Write your answer here._

---
# PART C: Inference optimization and production metrics

The rule for Part C is **change one thing at a time**. The prompts, `max_new_tokens=150` and the seed are fixed, so any
difference comes from the setting being tested. The model is the **16-bit base target model** from Part A.

## Step 1: Decoding strategies and sampling [3 marks] · *explainer §6*

In [ ]:
model = AutoModelForCausalLM.from_pretrained(CONFIG['target_model'], dtype=DTYPE16, device_map={'': 0})
run(model, 'warm-up')

STRATEGIES = {
    'Greedy':         dict(do_sample=False),
    'Beam (4)':       dict(do_sample=False, num_beams=4, early_stopping=True),
    'Top-K (k=50)':   dict(do_sample=True, top_k=50, top_p=1.0, temperature=1.0),
    'Top-P (p=0.9)':  dict(do_sample=True, top_k=0, top_p=0.9, temperature=1.0),
    'Temp 0.3':       dict(do_sample=True, top_k=0, top_p=1.0, temperature=0.3),
    'Temp 0.7':       dict(do_sample=True, top_k=0, top_p=1.0, temperature=0.7),
    'Temp 1.2':       dict(do_sample=True, top_k=0, top_p=1.0, temperature=1.2),
}
CONFIG['strategies'] = STRATEGIES

rows = []
for name, gen in STRATEGIES.items():
    for i, p in enumerate(DOMAIN_PROMPTS, 1):
        text, n, dt = run(model, p, gen)
        rows.append({'strategy': name, 'prompt': f'P{i}', 'text': text, 'new_tokens': n,
                     'seconds': round(dt, 2), 'tok_per_s': round(n / dt, 1)})
dec = pd.DataFrame(rows)
dec.to_csv(DATA_DIR / 'decoding_results.csv', index=False)

# the assignment's table: prompt x strategy (first 160 characters shown; full text is in the CSV)
table = dec.assign(cell=dec['text'].str.slice(0, 160) + ' … [' + dec['tok_per_s'].astype(str) + ' tok/s]') \
           .pivot(index='prompt', columns='strategy', values='cell')[list(STRATEGIES)]
display(table)
speed = dec.groupby('strategy', sort=False)[['tok_per_s', 'seconds']].mean().round(2)
speed

In [ ]:
# A simple diversity metric: distinct-2 = unique word pairs / all word pairs (1.0 = never repeats itself)
def distinct2(t):
    w = t.split(); bg = list(zip(w, w[1:]))
    return len(set(bg)) / max(1, len(bg))
dec['distinct_2'] = dec['text'].map(distinct2)
dec.groupby('strategy', sort=False)[['distinct_2', 'tok_per_s']].mean().round(3)

#### ✍️ Your inference (mandatory, write it in your own words)

- **100-word analysis:** which strategy would you deploy for an IT-helpdesk assistant, and why? Cite specific examples from the table (a command that was right or wrong, repetition, speed).

<details><summary>What to look for in your outputs</summary>

- Greedy: deterministic and fast, but a base model may loop (look for low distinct-2).
- Beam: often the most fluent, but slowest (4 hypotheses), and it can favour short, generic answers.
- Temp 1.2 / Top-K: more varied, and more invented flags. Dangerous when an admin will copy-paste the command.
- Factual domain → greedy or low temperature (0.3) is the usual pick. Name your choice and set BEST below.
</details>

_Write your answer here._

In [ ]:
BEST = 'Greedy'                 # <- change this after your analysis if you choose differently
BEST_GEN = STRATEGIES[BEST]
CONFIG['best_strategy'] = BEST

## Step 2: Speculative decoding [2 marks] · *explainer §7*

The **draft** (`Qwen2.5-0.5B`) proposes a few tokens, and the **target** (`Qwen2.5-1.5B`) checks them all in one forward pass.
Both share one tokenizer, which is required, and both fit in VRAM together (≈ 3.1 + 1.0 GB in 16-bit).

With greedy decoding, speculative decoding is **lossless**: the output should match plain greedy token for token. We check that.

In [ ]:
draft = AutoModelForCausalLM.from_pretrained(CONFIG['draft_model'], dtype=DTYPE16, device_map={'': 0})
assert draft.config.vocab_size == model.config.vocab_size, 'draft and target must share a vocabulary'
run(model, 'warm-up', BEST_GEN, assistant=draft)

def bench(m, prompts, gen, assistant=None, chat=False):
    rows = []
    for i, p in enumerate(prompts, 1):
        text, n, dt = run(m, p, gen, assistant=assistant, chat=chat)
        rows.append({'prompt': f'P{i}', 'text': text, 'new_tokens': n, 'seconds': dt})
    df = pd.DataFrame(rows)
    df.attrs['tok_per_s'] = df['new_tokens'].sum() / df['seconds'].sum()   # total tokens / total time
    return df

plain = bench(model, DOMAIN_PROMPTS, BEST_GEN)
spec  = bench(model, DOMAIN_PROMPTS, BEST_GEN, assistant=draft)
same  = [a == b for a, b in zip(plain['text'], spec['text'])]
spec_tbl = pd.DataFrame({
    'prompt': plain['prompt'],
    'plain s': plain['seconds'].round(2), 'speculative s': spec['seconds'].round(2),
    'speed-up': (plain['seconds'] / spec['seconds']).round(2),
    'identical output': same,
})
display(spec_tbl)
print(f"throughput: plain {plain.attrs['tok_per_s']:.1f} tok/s -> speculative {spec.attrs['tok_per_s']:.1f} tok/s "
      f"({spec.attrs['tok_per_s'] / plain.attrs['tok_per_s']:.2f}x)")
spec_tbl.to_csv(DATA_DIR / 'speculative_results.csv', index=False)

#### ✍️ Your inference (mandatory, write it in your own words)

- Report the throughput/latency change. Was it faster? By how much?
- Were outputs identical? If not, why might a few differ (16-bit rounding when verifying several tokens at once)?
- Why might the gain be small here? (Hint: the draft is 0.5B vs 1.5B, so how much cheaper is one draft step really? See explainer §7.)

<details><summary>What to look for in your outputs</summary>

- Speculative decoding helps when the draft is MUCH cheaper than the target and usually agrees with it. At batch 1 on small models, each step is dominated by fixed overhead (kernel launches for ~24–28 layers), so a 0.5B draft is not 3× cheaper in time.
- Gains are larger on predictable text (commands, boilerplate) than on creative text.
</details>

_Write your answer here._

## Step 3: 4-bit quantization and production cost [3 marks] · *explainer §8, §9*

Three configurations, each benchmarked **alone on the GPU** (everything else unloaded) on the **10 prompts** with the best strategy:

1. 16-bit baseline (the Part A model)
2. 4-bit NF4: `BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=…)`
3. 4-bit NF4 target + 16-bit draft (speculative decoding)

**Cost formula (from the assignment):** cost per 1M tokens = (1,000,000 ÷ throughput tok/s) ÷ 3600 × hourly rate.
The rate is 3.50/hr (T4) or 12.00/hr (A100). The currency symbol didn't survive the PDF text; the maths is the same in any currency.

In [ ]:
free('model', 'draft')

def load_16bit(mid):
    return AutoModelForCausalLM.from_pretrained(mid, dtype=DTYPE16, device_map={'': 0})
def load_4bit(mid):
    cfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=DTYPE16)
    return AutoModelForCausalLM.from_pretrained(mid, quantization_config=cfg, device_map={'': 0})

def measure(label, load_target, load_draft=None, repeats=2):
    gc.collect(); torch.cuda.empty_cache()
    target = load_target(CONFIG['target_model'])
    helper = load_draft(CONFIG['draft_model']) if load_draft else None
    weights_gb = torch.cuda.memory_allocated() / 1e9
    run(target, 'warm-up', BEST_GEN, assistant=helper)
    torch.cuda.reset_peak_memory_stats()
    tps = [bench(target, BENCH_PROMPTS, BEST_GEN, assistant=helper).attrs['tok_per_s'] for _ in range(repeats)]
    out = {'configuration': label, 'weights in VRAM (GB)': round(weights_gb, 2),
           'peak VRAM (GB)': round(torch.cuda.max_memory_allocated() / 1e9, 2),
           'throughput (tok/s)': round(statistics.median(tps), 1)}
    sample = run(target, BENCH_PROMPTS[0], BEST_GEN, assistant=helper)[0]
    del target, helper; gc.collect(); torch.cuda.empty_cache()
    return out, sample

results, samples = [], {}
for label, lt, ld in [('16-bit baseline (Part A)', load_16bit, None),
                      ('4-bit NF4', load_4bit, None),
                      ('4-bit NF4 + speculative (16-bit draft)', load_4bit, load_16bit)]:
    r, s = measure(label, lt, ld)
    results.append(r); samples[label] = s
    print(r)

cost = pd.DataFrame(results)
cost['cost / 1M tokens'] = ((1_000_000 / cost['throughput (tok/s)']) / 3600 * HOURLY_RATE).round(2)
cost['GPU-hours / 1M tokens'] = (1_000_000 / cost['throughput (tok/s)'] / 3600).round(2)
cost.to_csv(DATA_DIR / 'cost_table.csv', index=False)
print(f'GPU: {GPU_NAME}, rate {HOURLY_RATE}/hr, strategy {BEST}, {len(BENCH_PROMPTS)} prompts, median of 2 runs')
cost

In [ ]:
# quality check: the same prompt under each configuration (4-bit rounding can change wording)
pd.DataFrame(samples.items(), columns=['configuration', f'output for: {BENCH_PROMPTS[0]}'])

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
for ax, col in zip(axes, ['peak VRAM (GB)', 'throughput (tok/s)', 'cost / 1M tokens']):
    ax.barh(['16-bit', '4-bit', '4-bit + spec'], cost[col], color=['#64748b', '#3b82f6', '#8b5cf6'])
    ax.set_title(col); ax.invert_yaxis(); ax.grid(axis='x', alpha=.3)
plt.tight_layout(); plt.show()

#### ✍️ Your inference (mandatory, write it in your own words)

- **2–3 sentences:** which configuration would you deploy for this domain, citing the measured cost, VRAM and any quality difference?
- Did 4-bit make generation faster or slower? Explain why (explainer §8: memory saved vs dequantization work at batch size 1).

<details><summary>What to look for in your outputs</summary>

- 4-bit almost always cuts VRAM by about half or more. On a T4 at batch 1, bitsandbytes 4-bit is often SLOWER, because every matmul first unpacks the weights. Then the cost per token goes UP even though memory goes down.
- Pick for your constraint: if VRAM is the bottleneck (many models or adapters on one GPU, a cheaper GPU), 4-bit wins; if latency/cost per token is, 16-bit may win at this scale.
</details>

_Write your answer here._

---
## Run record and submission

The production habit: save **what was run** next to **what came out**, so the results can be reproduced and audited.

In [ ]:
(DATA_DIR / 'run_config.json').write_text(json.dumps(CONFIG, indent=1, default=str), encoding='utf-8')
shutil.make_archive(str(WORKDIR / 'domain_corpus'), 'zip', CORPUS_DIR)
print('Saved:')
for f in sorted(DATA_DIR.glob('*')) + [WORKDIR / 'domain_corpus.zip']:
    print(' ', f.relative_to(WORKDIR))

**Deliverables**
1. This notebook with outputs (Colab: *File → Download → .ipynb*).
2. HTML export: `jupyter nbconvert --to html practice.ipynb`.
3. `data/instruction_dataset.jsonl`.
4. `domain_corpus/*.txt` (zipped above).

Before exporting, check that every ✍️ cell has your own words. The 100-word decoding analysis (C1) and the deployment
recommendation (C3) carry the most marks.